# Локальный запуск DLightSB и DLightSB-MM

Ноутбук запускает baseline и MM отдельно для размерностей 2 16 и 64

Используется текущее локальное окружение и существующий checkout проекта

Ничего не скачивается и не устанавливается

## Локальное окружение

Перед запуском выберите kernel окружения проекта

Для GPU-запуска CUDA должна быть доступна из этого kernel

In [ ]:
from pathlib import Path
from time import perf_counter
import os
import subprocess
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "config.yaml").is_file() and (
            candidate / "src" / "run.py"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Не найден корень catsbench запустите ноутбук из папки проекта или notebooks"
    )


PROJECT_DIR = find_project_root(Path.cwd().resolve())
RUNS_DIR = PROJECT_DIR / "logs" / "runs"
SEED = 42

os.chdir(PROJECT_DIR)

print("Проект:", PROJECT_DIR)
print("Python:", sys.executable)
print("Логи:", RUNS_DIR)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна в выбранном kernel выберите CUDA-окружение проекта"
    )

print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## TensorBoard

Эту ячейку можно выполнить до или во время обучения

In [ ]:
from IPython import get_ipython


TENSORBOARD_LOG_DIR = RUNS_DIR.resolve()
tensorboard_args = (
    f'--logdir "{TENSORBOARD_LOG_DIR.as_posix()}" '
    '--reload_interval 5'
)

ipython = get_ipython()
ipython.run_line_magic("load_ext", "tensorboard")
ipython.run_line_magic("tensorboard", tensorboard_args)

## Функции запуска

Каждый вызов выполняет обучение затем test последнего checkpoint

Baseline и MM явно запускаются на одной GPU

In [ ]:
def run_streaming(command: list[str], env: dict[str, str]) -> None:
    print("Команда:", subprocess.list2cmdline(command))

    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

In [ ]:
def experiment_name(kind: str, dim: int) -> str:
    if kind == "baseline":
        return f"dlight_sb/benchmark_hd/d{dim}_g002"

    if kind == "mm":
        return f"dlight_sb_mm/benchmark_hd/d{dim}_g002"

    raise ValueError(f"Неизвестный алгоритм {kind}")


def latest_run_dir(experiment: str) -> Path:
    seed_dir = RUNS_DIR / experiment / str(SEED)
    candidates = [path for path in seed_dir.glob("*") if path.is_dir()]

    if not candidates:
        raise FileNotFoundError(f"Не найден запуск в {seed_dir}")

    return max(candidates, key=lambda path: path.name)


def run_experiment(kind: str, dim: int) -> Path:
    experiment = experiment_name(kind, dim)
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    env["PYTHONUNBUFFERED"] = "1"

    if kind == "mm":
        thread_count = str(os.cpu_count() or 1)
        env.setdefault("OMP_NUM_THREADS", thread_count)
        env.setdefault("MKL_NUM_THREADS", thread_count)
        env.setdefault("OPENBLAS_NUM_THREADS", thread_count)

    common_overrides = [
        f"experiment={experiment}",
        "trainer.accelerator=gpu",
        "trainer.devices=1",
        "logger=csv_tensorboard",
        "++callbacks.plotter_callback=null",
    ]

    train_command = [
        sys.executable,
        "-m",
        "src.run",
        *common_overrides,
        "logger.csv.version=train",
    ]
    test_command = [
        sys.executable,
        "-m",
        "src.run",
        "task_name=test",
        "ckpt_path=auto",
        *common_overrides,
        "logger.csv.version=test",
    ]

    started = perf_counter()
    print(f"Запуск {kind} d={dim}")
    run_streaming(train_command, env)
    run_streaming(test_command, env)

    run_dir = latest_run_dir(experiment)
    elapsed = perf_counter() - started
    print(f"Готово за {elapsed / 60:.1f} минут")
    print("Результат:", run_dir)
    return run_dir

# Размерность 2

## Baseline DLightSB d2

In [ ]:
baseline_d2_run = run_experiment("baseline", 2)

## DLightSB-MM d2

In [ ]:
mm_d2_run = run_experiment("mm", 2)

# Размерность 16

## Baseline DLightSB d16

In [ ]:
baseline_d16_run = run_experiment("baseline", 16)

## DLightSB-MM d16

In [ ]:
mm_d16_run = run_experiment("mm", 16)

# Размерность 64

## Baseline DLightSB d64

In [ ]:
baseline_d64_run = run_experiment("baseline", 64)

## DLightSB-MM d64

In [ ]:
mm_d64_run = run_experiment("mm", 64)

## Локальные результаты

Все checkpoints CSV TensorBoard Hydra-конфиги и логи остаются в `logs/runs`

Никакие архивы автоматически не создаются и не скачиваются